# E2E Type Progression

This notebook illustrates the typed interface progression used by `mellea-lrc`:

1. `Path` -> `PreprocessedDocument`
2. `PreprocessedDocument` -> `ExtractedDocument`
3. `ExtractedDocument` -> `ValidatedDocument`
4. `ValidatedDocument` -> `AssessedDocument`

Each step writes `local/snapshots/<doc>/<stage>.json`, reloads through `deserialize_*`, and reads its input from the previous snapshot. Re-run any step after setup as long as upstream snapshots exist.

In [ ]:
from pathlib import Path

from IPython.display import display

# Configuration
USE_BOOKMARKED = True  # True: bookmarked file; False: indexed test-data file
TEST_DATA_INDEX = 0
TEST_DATA_DIR = Path("../local/test_data")
BOOKMARKED_PATH = Path("../local/bookmarked/bookmarked.txt")
SNAPSHOT_ROOT = Path("../local/snapshots")
MELLEA_CONCURRENCY = 5

TEST_DATA_FILES = sorted(TEST_DATA_DIR.glob("*.txt"))
if USE_BOOKMARKED:
    if not BOOKMARKED_PATH.is_file():
        raise FileNotFoundError(f"Missing bookmarked file: {BOOKMARKED_PATH.resolve()}")
    DOC_PATH = BOOKMARKED_PATH
else:
    if not TEST_DATA_FILES:
        raise FileNotFoundError(f"No .txt files found in {TEST_DATA_DIR.resolve()}")
    if not 0 <= TEST_DATA_INDEX < len(TEST_DATA_FILES):
        raise IndexError(
            f"TEST_DATA_INDEX={TEST_DATA_INDEX} is outside 0..{len(TEST_DATA_FILES) - 1}"
        )
    DOC_PATH = TEST_DATA_FILES[TEST_DATA_INDEX]

SNAPSHOT_DIR = SNAPSHOT_ROOT / DOC_PATH.stem

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

display(
    {
        "doc_path": str(DOC_PATH),
        "snapshot_dir": str(SNAPSHOT_DIR),
    }
)

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path


from mellea_lrc.core.env import load_env_file

load_env_file(Path("../.env"), override=True)

from mellea_lrc.assessment import (
    CitationAssessmentResult,
    MelleaCallContext,
    run_assessment_async,
)
from mellea_lrc.extraction import run_extraction
from mellea_lrc.llm import llm_api_config_from_env
from mellea_lrc.preprocessing import run_preprocessing
from mellea_lrc.serialization import (
    deserialize_assessed_document,
    deserialize_extracted_document,
    deserialize_preprocessed_document,
    deserialize_validated_document,
    serialize_assessed_document,
    serialize_extracted_document,
    serialize_preprocessed_document,
    serialize_validated_document,
)
from mellea_lrc.validation import run_validation



from collections.abc import Callable

SNAPSHOT_STAGES = ("preprocessed", "extraction", "validation", "assessment")


def snapshot_path(stage: str) -> Path:
    if stage not in SNAPSHOT_STAGES:
        msg = f"Unknown snapshot stage: {stage}"
        raise ValueError(msg)
    return SNAPSHOT_DIR / f"{stage}.json"


def write_snapshot(stage: str, payload: object) -> Path:
    path = snapshot_path(stage)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    return path


def load_snapshot(stage: str, deserialize: Callable[[dict[str, object]], object]) -> object:
    path = snapshot_path(stage)
    if not path.exists():
        msg = f"Missing snapshot: {path}"
        raise FileNotFoundError(msg)
    payload = json.loads(path.read_text(encoding="utf-8"))
    return deserialize(payload)


def summarize_snapshot(stage: str, path: Path, payload: dict[str, object]) -> dict[str, object]:
    summary = {
        "stage": stage,
        "snapshot": str(path),
        "top_level_keys": sorted(payload.keys()),
    }
    counts = payload.get("counts")
    if isinstance(counts, dict):
        summary["counts"] = counts
    return summary

## Step 1: Preprocess

`run_preprocessing(Path)` produces the first typed boundary object: `PreprocessedDocument`. The snapshot is reloaded with `deserialize_preprocessed_document` before extraction.

In [ ]:
preprocessed_raw = run_preprocessing(DOC_PATH)
preprocessed_payload = serialize_preprocessed_document(preprocessed_raw)
preprocessed_path = write_snapshot("preprocessed", preprocessed_payload)
preprocessed = load_snapshot("preprocessed", deserialize_preprocessed_document)

display(type(preprocessed))
display(
    {
        **summarize_snapshot("preprocessed", preprocessed_path, preprocessed_payload),
        "chars": len(preprocessed.text),
        "source_path": preprocessed.source_metadata.path,
        "source_format": preprocessed.source_metadata.format.value,
        "backend": preprocessed.preprocessing_metadata.backend.value,
    }
)

## Step 2: Extract

`run_extraction(PreprocessedDocument)` advances the object to `ExtractedDocument`. The snapshot is reloaded with `deserialize_extracted_document` before validation.

In [ ]:
preprocessed = load_snapshot("preprocessed", deserialize_preprocessed_document)
extraction_raw = run_extraction(preprocessed)
extraction_payload = serialize_extracted_document(extraction_raw)
extraction_path = write_snapshot("extraction", extraction_payload)
extraction = load_snapshot("extraction", deserialize_extracted_document)

display(type(extraction))
display(summarize_snapshot("extraction", extraction_path, extraction_payload))
display(extraction_payload["citations"][:3])

## Step 3: Validate

`run_validation(ExtractedDocument)` advances the object to `ValidatedDocument`. The snapshot is reloaded with `deserialize_validated_document` before assessment.

This may call the configured CourtListener access service.

In [ ]:
extraction = load_snapshot("extraction", deserialize_extracted_document)
validation_raw = run_validation(extraction)
validation_payload = serialize_validated_document(validation_raw)
validation_path = write_snapshot("validation", validation_payload)
validation = load_snapshot("validation", deserialize_validated_document)

display(type(validation))
display(summarize_snapshot("validation", validation_path, validation_payload))
display(validation_payload["validations"][:3])

## Step 4: Assess

`run_assessment_async(ValidatedDocument)` advances the object to `AssessedDocument` and may call the configured LLM API in parallel. The snapshot is reloaded with `deserialize_assessed_document`.

In [ ]:
import time

validation = load_snapshot("validation", deserialize_validated_document)
llm_config = llm_api_config_from_env(os.environ)
mellea_started: dict[str, float] = {}


def on_mellea_call(ctx: MelleaCallContext) -> None:
    mellea_started[ctx.citation_id] = time.perf_counter()
    print(
        f"[mellea] start id={ctx.citation_id} "
        f"extracted={ctx.extracted_case_name!r}",
        flush=True,
    )


def on_mellea_done(ctx: MelleaCallContext, item: CitationAssessmentResult) -> None:
    elapsed = time.perf_counter() - mellea_started[ctx.citation_id]
    case = item.case_name.initial.status.value
    followup = item.case_name.followup.status.value
    print(
        f"[mellea] done  id={ctx.citation_id} "
        f"case={case} followup={followup} ({elapsed:.1f}s)",
        flush=True,
    )


assessment_raw = await run_assessment_async(
    validation,
    mellea_concurrency=MELLEA_CONCURRENCY,
    on_mellea_call=on_mellea_call,
    on_mellea_done=on_mellea_done,
)
assessment_payload = serialize_assessed_document(assessment_raw)
assessment_path = write_snapshot("assessment", assessment_payload)
assessment = load_snapshot("assessment", deserialize_assessed_document)
assessment_status_counts = {
    status: sum(
        item.status.value == status for item in assessment.assessments
    )
    for status in sorted({item.status.value for item in assessment.assessments})
}

display(
    {
        "assessment_path": str(assessment_path),
        "assessment_records": len(assessment.assessments),
        "assessment_status_counts": assessment_status_counts,
        "all_records_terminal": assessment.assessment_complete,
    }
)

display(type(assessment))
display(
    {
        **summarize_snapshot("assessment", assessment_path, assessment_payload),
        "api_base": llm_config.api_base,
        "model": llm_config.model,
        "response_format": llm_config.response_format.value,
        "certificate_verification": llm_config.cert_required,
        "assessment_records": len(assessment.assessments),
        "assessment_status_counts": assessment_status_counts,
        "all_records_terminal": assessment.assessment_complete,
    }
)